In [5]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("Importaciones correctas")

Python: c:\Users\Juan Palacios\Documents\Universidad\1_Semestre_Especializacion\GESTION_DE_DATOS\Proyecto_GYA\ProyectoFinalETL\venv\Scripts\python.exe
Pandas: 2.0.3
Importaciones correctas


In [6]:
from pathlib import Path

proyecto = Path(
    r"C:\Users\Juan Palacios\Documents\Universidad"
    r"\1_Semestre_Especializacion\GESTION_DE_DATOS"
    r"\Proyecto_GYA\ProyectoFinalETL"
)

archivo = proyecto / "data" / "bronze" / "ACCESOS_INTERNET_FIJO_2_8.csv"

print("Existe:", archivo.is_file())
print("Tamaño en MB:", round(archivo.stat().st_size / 1_000_000, 2))

Existe: True
Tamaño en MB: 441.33


In [7]:
lector = pd.read_csv(
    archivo,
    sep=";",
    encoding="utf-8",
    dtype="string",
    keep_default_na=False,
    na_values=[""],
    chunksize=100_000,
)

In [10]:
from collections import Counter

columnas_numericas = [
    "ANNO",
    "TRIMESTRE",
    "ACCESOS",
    "VELOCIDAD_EFECTIVA_DOWNSTREAM",
    "VELOCIDAD_EFECTIVA_UPSTREAM",
]

columnas_id = [
    "ID_EMPRESA",
    "ID_MUNICIPIO",
    "ID_DEPARTAMENTO",
    "ID_SEGMENTO",
    "ID_TECNOLOGIA",
]

# Reiniciar todos los acumuladores
total_filas = 0
total_bloques = 0

faltantes = Counter()
no_numericos = Counter()
controles = Counter()

ids_distintos = {col: set() for col in columnas_id}
resumenes = []
huellas = []

for bloque in lector:
    total_filas += len(bloque)
    total_bloques += 1

    # Guardar huellas para revisar posibles duplicados entre bloques
    huellas.append(
        pd.util.hash_pandas_object(bloque, index=False).to_numpy()
    )

    # Detectar valores faltantes y campos que contienen solo espacios
    vacios = bloque.isna() | bloque.apply(
        lambda col: col.str.strip().eq("").fillna(False)
    )

    faltantes.update(vacios.sum().to_dict())

    controles["filas_con_algun_faltante"] += int(
        vacios.any(axis=1).sum()
    )

    # Convertir números respetando la coma decimal
    numeros = bloque[columnas_numericas].apply(
        lambda col: pd.to_numeric(
            col.str.strip().str.replace(",", ".", regex=False),
            errors="coerce",
        )
    )

    # Separar textos no numéricos de campos realmente vacíos
    errores = numeros.isna() & ~vacios[columnas_numericas]
    no_numericos.update(errores.sum().to_dict())

    # Acumular identificadores únicos de todo el archivo
    for col in columnas_id:
        valores = bloque[col].dropna().str.strip()
        ids_distintos[col].update(valores[valores.ne("")])

    a = numeros["ACCESOS"]
    t = numeros["TRIMESTRE"]
    y = numeros["ANNO"]

    down = numeros["VELOCIDAD_EFECTIVA_DOWNSTREAM"]
    up = numeros["VELOCIDAD_EFECTIVA_UPSTREAM"]

    # Controles sobre accesos
    controles["accesos_negativos"] += int(a.lt(0).sum())
    controles["accesos_en_cero"] += int(a.eq(0).sum())

    controles["accesos_fraccionarios"] += int(
        (a.notna() & a.mod(1).ne(0)).sum()
    )

    # Controles sobre trimestres
    controles["trimestres_fuera_de_1_a_4"] += int(
        (t.notna() & ~t.isin([1, 2, 3, 4])).sum()
    )

    # Controles sobre velocidades
    controles["descarga_negativa"] += int(down.lt(0).sum())
    controles["descarga_en_cero"] += int(down.eq(0).sum())
    controles["subida_negativa"] += int(up.lt(0).sum())
    controles["subida_en_cero"] += int(up.eq(0).sum())

    # Conservar y contar el texto NA de tecnología por separado
    controles["tecnologia_texto_NA"] += int(
        bloque["TECNOLOGIA"].str.strip().eq("NA").sum()
    )

    # Período válido: año entero positivo y trimestre entre 1 y 4
    periodo_valido = (
        y.gt(0)
        & y.mod(1).eq(0)
        & t.isin([1, 2, 3, 4])
    ).fillna(False)

    # Accesos válidos para sumar: enteros y no negativos
    accesos_validos = (
        a.ge(0) & a.mod(1).eq(0)
    ).fillna(False)

    controles["filas_con_periodo_invalido"] += int(
        (~periodo_valido).sum()
    )

    controles["filas_con_accesos_invalidos_o_faltantes"] += int(
        (~accesos_validos).sum()
    )

    # Resumen del bloque por año y trimestre
    temporal = numeros.loc[
        periodo_valido, ["ANNO", "TRIMESTRE", "ACCESOS"]
    ].copy()

    temporal["ACCESOS"] = temporal["ACCESOS"].where(accesos_validos)

    parcial = temporal.groupby(["ANNO", "TRIMESTRE"]).agg(
        registros=("ACCESOS", "size"),
        registros_con_accesos_validos=("ACCESOS", "count"),
        accesos=("ACCESOS", "sum"),
    )

    resumenes.append(parcial)

    # Mostrar progreso cada cinco bloques
    if total_bloques % 5 == 0:
        print(
            f"Procesados {total_bloques} bloques: "
            f"{total_filas:,} filas"
        )

# Tabla global de faltantes
calidad_columnas = pd.DataFrame({
    "faltantes": dict(faltantes)
})

calidad_columnas["porcentaje_faltante"] = (
    calidad_columnas["faltantes"] / total_filas * 100
).round(4)

# Unir los pequeños resúmenes y sumar períodos repetidos
resumen_periodos = (
    pd.concat(resumenes)
    .groupby(level=["ANNO", "TRIMESTRE"])
    .sum()
    .sort_index()
    .reset_index()
)

resumen_periodos[["ANNO", "TRIMESTRE"]] = (
    resumen_periodos[["ANNO", "TRIMESTRE"]].astype(int)
)

# Evitar presentar cero cuando un período no tiene accesos válidos
resumen_periodos["accesos"] = (
    resumen_periodos["accesos"]
    .where(resumen_periodos["registros_con_accesos_validos"].gt(0))
    .astype("Int64")
)

# Revisar huellas repetidas en todo el archivo
duplicados_posibles = int(
    pd.Series(np.concatenate(huellas)).duplicated().sum()
)

del huellas

print(
    f"Finalizado: {total_bloques} bloques "
    f"y {total_filas:,} filas"
)

Procesados 5 bloques: 500,000 filas
Procesados 10 bloques: 1,000,000 filas
Procesados 15 bloques: 1,500,000 filas
Procesados 20 bloques: 2,000,000 filas
Procesados 25 bloques: 2,500,000 filas
Procesados 30 bloques: 3,000,000 filas
Finalizado: 33 bloques y 3,288,251 filas


In [11]:
print("FALTANTES POR COLUMNA")
display(calidad_columnas)

print("VALORES QUE NO SE PUDIERON CONVERTIR A NÚMEROS")
display(
    pd.Series(dict(no_numericos), name="cantidad")
    .to_frame()
)

print("CONTROLES DE CALIDAD")
display(
    pd.Series(dict(controles), name="cantidad")
    .to_frame()
)

print("IDENTIFICADORES DISTINTOS EN TODO EL ARCHIVO")
display(
    pd.Series(
        {col: len(valores) for col, valores in ids_distintos.items()},
        name="cantidad",
    ).to_frame()
)

print("POSIBLES FILAS DUPLICADAS:", duplicados_posibles)

print("REGISTROS Y ACCESOS POR AÑO Y TRIMESTRE")
display(resumen_periodos)

FALTANTES POR COLUMNA


,faltantes,porcentaje_faltante
ACCESOS,0,0.0
ANNO,0,0.0
DEPARTAMENTO,0,0.0
EMPRESA,0,0.0
ID_DEPARTAMENTO,0,0.0
ID_EMPRESA,0,0.0
ID_MUNICIPIO,0,0.0
ID_SEGMENTO,0,0.0
ID_TECNOLOGIA,0,0.0
MUNICIPIO,0,0.0


VALORES QUE NO SE PUDIERON CONVERTIR A NÚMEROS


,cantidad
ANNO,0
TRIMESTRE,0
ACCESOS,0
VELOCIDAD_EFECTIVA_DOWNSTREAM,0
VELOCIDAD_EFECTIVA_UPSTREAM,0


CONTROLES DE CALIDAD


,cantidad
filas_con_algun_faltante,0
accesos_negativos,0
accesos_en_cero,5360
accesos_fraccionarios,0
trimestres_fuera_de_1_a_4,0
descarga_negativa,0
descarga_en_cero,2380
subida_negativa,0
subida_en_cero,2779
tecnologia_texto_NA,129


IDENTIFICADORES DISTINTOS EN TODO EL ARCHIVO


,cantidad
ID_EMPRESA,2028
ID_MUNICIPIO,1124
ID_DEPARTAMENTO,34
ID_SEGMENTO,10
ID_TECNOLOGIA,19


POSIBLES FILAS DUPLICADAS: 0
REGISTROS Y ACCESOS POR AÑO Y TRIMESTRE


,ANNO,TRIMESTRE,registros,registros_con_accesos_validos,accesos
0,2017,2,48570,48570,6193671
1,2017,3,50539,50539,6284505
2,2017,4,51901,51901,6352946
3,2018,1,55048,55048,6479592
4,2018,2,56975,56975,6578723
5,2018,3,59331,59331,6660174
6,2018,4,62368,62368,6731666
7,2019,1,66175,66175,6801044
8,2019,2,71886,71886,6911288
9,2019,3,77249,77249,7013184


In [8]:
def normalizar_codigo(serie, longitud):
    texto = serie.astype("string").str.strip()

    valido = (
        texto.str.fullmatch(rf"[0-9]{{1,{longitud}}}", na=False)
        & ~texto.str.fullmatch(r"0+", na=False)
    )

    return texto.where(valido).str.zfill(longitud)


ruta_divipola = proyecto / "data" / "bronze" / "DIVIPOLA-_C_digos_municipios.csv"

divipola = pd.read_csv(
    ruta_divipola,
    sep=",",
    encoding="utf-8-sig",
    dtype="string",
).rename(columns={
    "Código Municipio": "codigo_municipio_div",
    "Nombre Municipio": "municipio_div",
    "Código Departamento": "codigo_departamento_div",
    "Nombre Departamento": "departamento_div",
    "Tipo: Municipio / Isla / Área no municipalizada": "tipo_territorio_div",
})

campos_div = [
    "codigo_municipio_div",
    "municipio_div",
    "codigo_departamento_div",
    "departamento_div",
    "tipo_territorio_div",
]

divipola = divipola[campos_div].copy()

divipola["codigo_municipio_div"] = normalizar_codigo(
    divipola["codigo_municipio_div"], 5
)

divipola["codigo_departamento_div"] = normalizar_codigo(
    divipola["codigo_departamento_div"], 2
)

# Comprobar que el catálogo permite un cruce seguro
if divipola[
    ["codigo_municipio_div", "codigo_departamento_div"]
].isna().any().any():
    raise ValueError("DIVIPOLA contiene códigos vacíos o con formato no válido.")

if divipola["codigo_municipio_div"].duplicated().any():
    raise ValueError("DIVIPOLA tiene códigos municipales repetidos.")

print("Códigos territoriales en DIVIPOLA:", len(divipola))
display(divipola.head())

Códigos territoriales en DIVIPOLA: 1122


,codigo_municipio_div,municipio_div,codigo_departamento_div,departamento_div,tipo_territorio_div
0,05001,MEDELLÍN,05,ANTIOQUIA,Municipio
1,05002,ABEJORRAL,05,ANTIOQUIA,Municipio
2,05004,ABRIAQUÍ,05,ANTIOQUIA,Municipio
3,05021,ALEJANDRÍA,05,ANTIOQUIA,Municipio
4,05030,AMAGÁ,05,ANTIOQUIA,Municipio


In [9]:
campos = [
    "ANNO",
    "TRIMESTRE",
    "ID_MUNICIPIO",
    "MUNICIPIO",
    "ID_DEPARTAMENTO",
    "DEPARTAMENTO",
]

partes_territoriales = []

lector_territorial = pd.read_csv(
    archivo,
    sep=";",
    encoding="utf-8",
    dtype="string",
    usecols=campos,
    keep_default_na=False,
    na_values=[""],
    chunksize=100_000,
)

for bloque in lector_territorial:
    parcial = (
        bloque.groupby(campos, dropna=False)
        .size()
        .reset_index(name="registros")
    )

    partes_territoriales.append(parcial)

territorios = (
    pd.concat(partes_territoriales, ignore_index=True)
    .groupby(campos, dropna=False, as_index=False)["registros"]
    .sum()
)

del partes_territoriales

# Crear columnas nuevas; conservar los códigos originales
territorios["codigo_municipio"] = normalizar_codigo(
    territorios["ID_MUNICIPIO"], 5
)

territorios["codigo_departamento"] = normalizar_codigo(
    territorios["ID_DEPARTAMENTO"], 2
)

print(
    "Registros originales representados:",
    int(territorios["registros"].sum()),
)

display(territorios.head())

Registros originales representados: 3288251


,ANNO,TRIMESTRE,ID_MUNICIPIO,MUNICIPIO,ID_DEPARTAMENTO,DEPARTAMENTO,registros,codigo_municipio,codigo_departamento
0,2017,2,0,COLOMBIA,0,COLOMBIA,1,<NA>,<NA>
1,2017,2,11001,"BOGOTÁ, D.C.",11,BOGOTÁ D.C.,1383,11001,11
2,2017,2,13001,CARTAGENA,13,BOLÍVAR,637,13001,13
3,2017,2,13006,ACHÍ,13,BOLÍVAR,13,13006,13
4,2017,2,13030,ALTOS DEL ROSARIO,13,BOLÍVAR,7,13030,13


In [10]:
validacion_territorial = territorios.merge(
    divipola,
    how="left",
    left_on="codigo_municipio",
    right_on="codigo_municipio_div",
    validate="many_to_one",
    indicator=True,
)

coincide = validacion_territorial["_merge"].eq("both")

es_nacional = (
    validacion_territorial["ID_MUNICIPIO"]
    .str.strip()
    .str.fullmatch(r"0+", na=False)
)

# Clasificar el resultado del cruce municipal
validacion_territorial["estado"] = "codigo_no_en_DIVIPOLA"

validacion_territorial.loc[
    coincide, "estado"
] = "coincide_con_DIVIPOLA"

validacion_territorial.loc[
    validacion_territorial["codigo_municipio"].isna(), "estado"
] = "formato_no_valido"

validacion_territorial.loc[
    es_nacional, "estado"
] = "codigo_0_COLOMBIA"

# Comprobar el departamento de los municipios encontrados
validacion_territorial["departamento_coincide"] = pd.Series(
    pd.NA,
    index=validacion_territorial.index,
    dtype="boolean",
)

validacion_territorial.loc[coincide, "departamento_coincide"] = (
    validacion_territorial.loc[coincide, "codigo_departamento"]
    .eq(
        validacion_territorial.loc[
            coincide, "codigo_departamento_div"
        ]
    )
    .fillna(False)
)

resumen_territorial = validacion_territorial.groupby("estado").agg(
    registros_csv=("registros", "sum"),
    codigos_originales=("ID_MUNICIPIO", "nunique"),
    codigos_normalizados=("codigo_municipio", "nunique"),
)

sin_coincidencia = validacion_territorial.loc[
    validacion_territorial["estado"].isin([
        "codigo_no_en_DIVIPOLA",
        "formato_no_valido",
    ])
]

registros_nacionales = validacion_territorial.loc[es_nacional]

departamentos_diferentes = validacion_territorial.loc[
    coincide
    & validacion_territorial["departamento_coincide"].eq(False)
]

print("RESUMEN DEL CRUCE")
display(resumen_territorial)

print("CÓDIGOS SIN COINCIDENCIA")
display(sin_coincidencia[campos + ["registros", "estado"]])

print("REGISTROS CON CÓDIGO 0 = COLOMBIA")
display(registros_nacionales[campos + ["registros"]])

print("MUNICIPIOS CUYO DEPARTAMENTO NO COINCIDE")
display(
    departamentos_diferentes[
        campos + ["codigo_departamento_div", "departamento_div", "registros"]
    ]
)

RESUMEN DEL CRUCE


,registros_csv,codigos_originales,codigos_normalizados
estado,,,
codigo_0_COLOMBIA,3,1,0
codigo_no_en_DIVIPOLA,17,1,1
coincide_con_DIVIPOLA,3288231,1122,1122


CÓDIGOS SIN COINCIDENCIA


,ANNO,TRIMESTRE,ID_MUNICIPIO,MUNICIPIO,ID_DEPARTAMENTO,DEPARTAMENTO,registros,estado
1539,2017,3,27086,BELEN DE BAJIRA,27,CHOCÓ,1,codigo_no_en_DIVIPOLA
4856,2018,2,27086,BELEN DE BAJIRA,27,CHOCÓ,1,codigo_no_en_DIVIPOLA
5967,2018,3,27086,BELEN DE BAJIRA,27,CHOCÓ,1,codigo_no_en_DIVIPOLA
7079,2018,4,27086,BELEN DE BAJIRA,27,CHOCÓ,1,codigo_no_en_DIVIPOLA
8187,2019,1,27086,BELEN DE BAJIRA,27,CHOCÓ,1,codigo_no_en_DIVIPOLA
10412,2019,3,27086,BELEN DE BAJIRA,27,CHOCÓ,2,codigo_no_en_DIVIPOLA
11528,2019,4,27086,BELEN DE BAJIRA,27,CHOCÓ,2,codigo_no_en_DIVIPOLA
12644,2020,1,27086,BELEN DE BAJIRA,27,CHOCÓ,2,codigo_no_en_DIVIPOLA
13759,2020,2,27086,BELEN DE BAJIRA,27,CHOCÓ,2,codigo_no_en_DIVIPOLA
17103,2021,1,27086,BELEN DE BAJIRA,27,CHOCÓ,2,codigo_no_en_DIVIPOLA


REGISTROS CON CÓDIGO 0 = COLOMBIA


,ANNO,TRIMESTRE,ID_MUNICIPIO,MUNICIPIO,ID_DEPARTAMENTO,DEPARTAMENTO,registros
0,2017,2,0,COLOMBIA,0,COLOMBIA,1
1105,2017,3,0,COLOMBIA,0,COLOMBIA,2


MUNICIPIOS CUYO DEPARTAMENTO NO COINCIDE


,ANNO,TRIMESTRE,ID_MUNICIPIO,MUNICIPIO,ID_DEPARTAMENTO,DEPARTAMENTO,codigo_departamento_div,departamento_div,registros


In [12]:
def normalizar_nombre(serie):
    return (
        serie.astype("string")
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("ascii")
        .str.upper()
        .str.replace(r"[^A-Z0-9]+", " ", regex=True)
        .str.strip()
        .astype("string")  # Mantener un tipo compatible con valores <NA>
    )


coincide = validacion_territorial["_merge"].eq("both")

nombre_internet = normalizar_nombre(
    validacion_territorial["MUNICIPIO"]
)

nombre_divipola = normalizar_nombre(
    validacion_territorial["municipio_div"]
)

mascara_diferencias = (
    coincide & nombre_internet.ne(nombre_divipola)
).fillna(False)

nombres_diferentes = validacion_territorial.loc[
    mascara_diferencias
]

comparacion_nombres = nombres_diferentes[
    ["codigo_municipio", "MUNICIPIO", "municipio_div"]
].drop_duplicates()

print("Combinaciones diferentes:", len(comparacion_nombres))
print(
    "Códigos territoriales involucrados:",
    comparacion_nombres["codigo_municipio"].nunique(),
)

display(comparacion_nombres)

Combinaciones diferentes: 35
Códigos territoriales involucrados: 32


,codigo_municipio,MUNICIPIO,municipio_div
2,13001,CARTAGENA,CARTAGENA DE INDIAS
90,15332,GÜICÁN,GÜICÁN DE LA SIERRA
232,19418,LÓPEZ,LÓPEZ DE MICAY
271,20443,MANAURE,MANAURE BALCÓN DEL CESAR
300,23586,PURÍSIMA,PURÍSIMA DE LA CONCEPCIÓN
391,25662,SAN JUAN DE RÍO SECO,SAN JUAN DE RIOSECO
513,47161,CERRO SAN ANTONIO,CERRO DE SAN ANTONIO
548,50223,SAN LUIS DE CUBARRAL,CUBARRAL
567,05042,SANTAFÉ DE ANTIOQUIA,SANTA FÉ DE ANTIOQUIA
677,52835,TUMACO - SAN ANDRES DE TUMACO,SAN ANDRÉS DE TUMACO
